# 📘 S9_P5 — Des **vraies** données : Air Quality (UCI)

> **Comment lire ce notebook.** Le code est **exactement celui que tu as écrit**. Je n'ai
> ajouté que des explications. L'original est dans `01_Exercices_originaux/S9_P5_DL.ipynb`.
>
> ⚠️ **Ce notebook est en cours d'écriture** (6 cellules : chargement et premier nettoyage).
> Le cours ci-dessous couvre ce que tu as fait et te prépare la suite.

## 🎯 Le vrai saut du P5

| | S9_P4 | **S9_P5** |
|---|---|---|
| Données | fabriquées par toi | **relevés de capteurs réels** |
| La vérité | tu la connais (`sin(x1) + 0.5·x2²`) | **personne ne la connaît** |
| Valeurs manquantes | aucune | **partout, et déguisées** |
| Bruit | 0,15, choisi par toi | inconnu |
| Le travail | modéliser | **nettoyer**, puis modéliser |

En S9_P4, si ça ratait, c'était le modèle — les données étaient parfaites par construction.
**Ici, tu perds ce luxe.** Et tu vas découvrir la vérité du métier : *l'essentiel du travail
d'un data scientist, c'est le nettoyage.* Le réseau, c'est la partie facile.

## 📊 Le dataset

Relevés horaires d'une station de mesure de qualité de l'air en Italie, **mars 2004 → avril
2005**, ~9 400 lignes. Un classique de l'UCI Machine Learning Repository.

| Type de colonne | Exemples | C'est quoi |
|---|---|---|
| **`(GT)`** = *Ground Truth* | `CO(GT)`, `NOx(GT)`, `NO2(GT)` | la **vraie** concentration, mesurée par un analyseur de référence |
| **`PT08.Sx`** | `PT08.S1(CO)`, `PT08.S5(O3)` | la réponse brute d'un **capteur bon marché** |
| **Météo** | `T`, `RH`, `AH` | température, humidité relative et absolue |

> 💡 **L'intérêt du jeu de données est là.** Les `PT08` sont des capteurs à quelques euros,
> les `(GT)` viennent d'un appareil de laboratoire hors de prix. **Peut-on prédire la mesure
> chère à partir des capteurs pas chers ?** Si oui, on équipe une ville entière pour le prix
> d'une station. C'est un vrai problème d'ingénieur, et c'est ta cible naturelle.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.loggers import CSVLogger

pl.seed_everything(32)

Seed set to 32


32

## 🧠 THÉORIE — Charger un CSV européen (et un fichier qui ment)

### 🎭 `AirQuality.xls` n'est **pas** un fichier Excel

Ouvre-le avec un éditeur de texte, tu verras :

```
Date;Time;CO(GT);PT08.S1(CO);NMHC(GT);...;;
10/03/2004;18.00.00;2,6;1360;150;11,9;...;;
```

**C'est du texte brut.** Un vrai `.xls` commence par la signature binaire
`D0 CF 11 E0` (format OLE2 de Microsoft) ; celui-ci commence par `Date;Tim`. **Seule
l'extension ment.**

C'est pour ça que ton `pd.read_csv(...)` est **la bonne méthode** — et pas
`pd.read_excel(...)`, qui planterait sur `Excel file format cannot be determined`. Bonne
nouvelle au passage : tu évites la dépendance `xlrd`, qu'il aurait fallu installer.

> 📌 **Le réflexe :** ne fais jamais confiance à une extension. En cas de doute, ouvre les
> premiers octets. Ici, ton code prouve que tu as regardé le contenu — c'est ce qu'il faut
> faire.

### Les deux arguments qui font tout

```python
df = pd.read_csv(datapath, sep=";", decimal=",")
```

| Argument | Pourquoi il est indispensable |
|---|---|
| `sep=";"` | c'est un CSV **européen** : la virgule sert aux décimales, donc le séparateur est `;` |
| `decimal=","` | `2,6` doit devenir le nombre `2.6` — **sans ça, pandas lit du texte** |

### 🔴 Ce qui se passe si tu oublies `decimal=","`

Pandas **ne plante pas**. Il lit `"2,6"` comme une **chaîne de caractères**. Ta colonne
devient `object` au lieu de `float64`, et tout casse plus loin : `.mean()` échoue, le
`StandardScaler` refuse, `torch.tensor()` proteste. Le message d'erreur, lui, arrivera vingt
cellules plus tard et ne parlera pas du tout de virgules.

**Le réflexe qui te sauve : `df.dtypes` juste après un `read_csv`.** Si une colonne
numérique est en `object`, tu as un problème de format, pas de données.

### ✅ Ton chemin est le bon

```python
datapath = "../Data/AirQuality.xls"
```

Le `../` remonte de `01_Exercices_originaux/` vers `Deep_learning/`, puis redescend dans
`Data/`. C'est **exactement la convention du dépôt**, la même que tes notebooks ML
(`pd.read_csv("../Data/Advertising.csv")`). Ton notebook s'exécute directement après un
clone, chez n'importe qui.

In [3]:
datapath = "../Data/AirQuality.xls"

df = pd.read_csv(datapath, sep=";", decimal=",")

## 🧠 Les colonnes fantômes `Unnamed: 15` et `Unnamed: 16`

D'où sortent-elles ? Regarde la fin de chaque ligne du fichier :

```
10/03/2004;18.00.00;2,6;1360;...;0,7578;;
                                       ^^  deux points-virgules pour rien
```

Le fichier a été exporté avec **deux séparateurs en trop** à la fin de chaque ligne. Pandas
fait son travail : deux séparateurs = deux colonnes de plus. Comme l'en-tête ne leur donne
pas de nom, il invente `Unnamed: 15` et `Unnamed: 16`. Elles sont **entièrement vides**.

```python
df = df.drop(columns=["Unnamed: 15", "Unnamed: 16"])
```

Ta suppression est correcte. ✅

> 💡 **La variante générique**, utile quand tu ne sais pas combien il y en a :
> ```python
> df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
> ```
> ou, dès le chargement : `pd.read_csv(..., usecols=range(15))`.

In [4]:
df = df.drop(columns=["Unnamed: 15", "Unnamed: 16"])

## 🔴 THÉORIE — `dropna()` te donne une **fausse** impression de propreté

**La cellule la plus importante du notebook, et le piège n°1 de ce dataset.**

### Ce que `dropna()` enlève vraiment

Ton fichier se termine par **114 lignes complètement vides** (encore l'export bancal). Ce
sont elles que `dropna()` supprime :

| | lignes | colonnes |
|---|---|---|
| après `read_csv` | 9471 | 17 |
| après `drop` des Unnamed | 9471 | 15 |
| **après `dropna()`** | **9357** | 15 |

9471 − 9357 = **114 lignes vides supprimées**. Parfait.

### 🔴 Sauf que le dataset est encore **truffé** de valeurs manquantes

Voici le piège, et il est vicieux : **dans ce dataset, une valeur manquante n'est pas un
`NaN`. C'est un `-200`.** Les auteurs ont codé l'absence de mesure par une valeur sentinelle.

`dropna()` cherche des `NaN`. Il ne voit **rien**. **Mesuré sur ton `df` après ton
`dropna()` :**

| Colonne | Cellules à −200 | Part |
|---|---|---|
| **`NMHC(GT)`** | 8443 / 9357 | 🔴 **90,2 %** |
| `CO(GT)` | 1683 / 9357 | 18,0 % |
| `NOx(GT)` | 1639 / 9357 | 17,5 % |
| `NO2(GT)` | 1642 / 9357 | 17,5 % |
| toutes les autres | 366 / 9357 | 3,9 % |
| | **16 701 au total** | |

**Ton DataFrame « propre » contient 16 701 valeurs manquantes déguisées en nombres.**

### Pourquoi c'est grave

Une concentration de gaz **ne peut pas** être négative. Ces `-200` sont physiquement
absurdes. Si tu les laisses passer :

- ton `StandardScaler` calcule un μ et un σ **complètement faussés** par ces −200
- ton réseau apprend que « −200 » est une valeur normale à prédire
- tes courbes descendront, ton R² sera correct, et **ton modèle sera bon à jeter**

**Et rien ne plantera.** C'est exactement le type de bug qui passe une revue de code et
ruine un projet.

### La correction : dire à pandas que −200 veut dire « manquant »

```python
df = df.replace(-200, np.nan)     # on traduit la sentinelle en vrai NaN
df = df.dropna()                  # MAINTENANT dropna() fait son travail
```

ou, encore mieux, dès le chargement :

```python
df = pd.read_csv(datapath, sep=";", decimal=",", na_values=[-200])
```

### ⚠️ Puis la vraie décision : `NMHC(GT)` est à jeter

Avec **90 % de manquants**, cette colonne n'a rien à t'apprendre. Deux options, et la
première est la bonne :

| Option | Conséquence |
|---|---|
| ✅ **supprimer la colonne** `NMHC(GT)` | tu gardes tes ~9000 lignes |
| ❌ `dropna()` en la gardant | tu perds **90 % de tes lignes** — il t'en reste ~900 |

```python
df = df.drop(columns=["NMHC(GT)"])   # 90 % de vide : elle ne sert a rien
df = df.replace(-200, np.nan).dropna()
```

> 📌 **La leçon du S9_P5, et elle vaut pour toute ta carrière :** avant de faire confiance à
> `dropna()`, **demande-toi comment _ce_ dataset code ses manquants**. `-200`, `-999`, `0`,
> `9999`, `"N/A"`, `""`, `"?"` — chaque domaine a sa convention. Le README du dataset le dit.
> **Lis-le.** Un `df.describe()` te met aussi la puce à l'oreille : un minimum à −200 sur une
> concentration, ça ne trompe personne.

In [5]:
df = df.dropna()

### 👀 Regarder le DataFrame

Afficher `df` est le bon réflexe. **Mais fais-en trois de plus**, ils valent bien mieux :

```python
df.dtypes         # tout est bien numerique ? (sinon -> probleme de decimal=",")
df.describe()     # 👈 LE plus utile : un min a -200 saute aux yeux
df.isna().sum()   # combien de NaN par colonne
```

`describe()` t'aurait montré le piège immédiatement :

| | `CO(GT)` | `NMHC(GT)` |
|---|---|---|
| **min** | **-200** 🔴 | **-200** 🔴 |
| max | 11,9 | 1189 |

Un minimum de −200 sur une concentration de monoxyde de carbone, c'est **physiquement
impossible**. Le tableau te le crie.

In [6]:
df

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9352,04/04/2005,10.00.00,3.1,1314.0,-200.0,13.5,1101.0,472.0,539.0,190.0,1374.0,1729.0,21.9,29.3,0.7568
9353,04/04/2005,11.00.00,2.4,1163.0,-200.0,11.4,1027.0,353.0,604.0,179.0,1264.0,1269.0,24.3,23.7,0.7119
9354,04/04/2005,12.00.00,2.4,1142.0,-200.0,12.4,1063.0,293.0,603.0,175.0,1241.0,1092.0,26.9,18.3,0.6406
9355,04/04/2005,13.00.00,2.1,1003.0,-200.0,9.5,961.0,235.0,702.0,156.0,1041.0,770.0,28.3,13.5,0.5139


---

# 🎓 Ce qu'il faut retenir du S9_P5

### Ce que tu as fait ✅

| Étape | Statut |
|---|---|
| Repérer que le `.xls` est un CSV déguisé | ✅ `read_csv` et pas `read_excel` |
| `sep=";"` + `decimal=","` — le CSV européen | ✅ |
| Chemin `../Data/` — la convention du dépôt | ✅ |
| Virer les colonnes fantômes `Unnamed` | ✅ |
| `dropna()` — les 114 lignes vides | ✅ mais **insuffisant** |

### 🔴 Ce qu'il reste à faire **avant** de modéliser

```python
df = df.drop(columns=["NMHC(GT)"])       # 90 % de manquants : inexploitable
df = df.replace(-200, np.nan).dropna()   # les manquants deguises
```

**Sans ces deux lignes, tout ce que tu construiras derrière sera faux.** Pas cassé — faux,
ce qui est pire.

### La feuille de route pour la suite

Le S9_P4 t'a donné le squelette. Il ne change pas, tu ne fais que le remplir :

```
1. nettoyer      → -200 -> NaN, virer NMHC(GT), dropna    ← LA vraie difficulte
2. choisir       → features = les PT08 (capteurs pas chers)
                   cible    = CO(GT) (l'analyseur de reference)
3. split         → train_test_split(..., random_state=32)
4. scaler        → fit sur le TRAIN seulement, transform ailleurs
5. tenseurs      → float32, formes (n, features)
6. Dataset       → ton RegressionDataset du P4 marche tel quel
7. DataLoader    → shuffle=True / False
8. MLPRegressor  → change juste input_dim
9. Trainer       → un objet neuf par run
10. evaluer      → eval() + no_grad() + inverse_transform + (y_true, y_pred)
```

### ⏰ Un piège qui t'attend : ce sont des **séries temporelles**

Tes données sont des relevés **horaires successifs**. Un `train_test_split` classique tire au
hasard, donc la mesure de 14 h part en train et celle de 15 h en validation. Elles sont
quasi identiques (l'air ne change pas en une heure) : **ton modèle a vu la réponse**, et ton
score sera trop beau.

Pour une vraie évaluation, coupe **dans le temps** : les 80 premiers % en train, les 20
derniers en validation.

```python
n = int(len(df) * 0.8)
train, val = df[:n], df[n:]     # pas de shuffle : on respecte la chronologie
```

> 💡 C'est un sujet à part entière (`TimeSeriesSplit` en sklearn). Pour un premier essai, le
> split aléatoire passe — **mais sache que ton score sera optimiste**, et sache pourquoi.

### Les deux réflexes à emporter du P5

1. **`df.describe()` juste après le chargement.** Toujours. Un min ou un max absurde t'évite
   des jours de travail sur des données pourries.
2. **Demande-toi comment ce dataset code ses manquants** avant de faire confiance à
   `dropna()`. `-200` ici, `-999` ailleurs, `0` parfois. Le README le dit.